# Web Scraper Showdown: BeautifulSoup vs Scrapy vs Crawl4AI

A practical comparison for AI-driven workflows. We'll scrape a **JavaScript-rendered** page and compare:

- **Lines of Code** required
- **Ease of Use** for developers
- **AI-Readiness** of the output

**Target:** `https://quotes.toscrape.com/js/` — quotes rendered entirely via JavaScript.


In [1]:
# Setup: pip install beautifulsoup4 playwright scrapy crawl4ai pandas nest_asyncio
# Then: playwright install chromium

import asyncio
import time
import nest_asyncio
import pandas as pd

nest_asyncio.apply()  # Enable nested async in Jupyter

TARGET_URL = "https://quotes.toscrape.com/js/"

---

## 1. BeautifulSoup: Manual Parsing & Cleaning

BeautifulSoup **cannot execute JavaScript**. For JS-heavy pages, we need:

1. A browser automation tool (Playwright) to render the page
2. Manual HTML parsing with CSS selectors
3. Manual text cleaning and formatting

This is the **traditional approach** — powerful but labor-intensive.


In [ ]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


async def scrape_with_beautifulsoup(url: str) -> dict:
    """BeautifulSoup requires external JS rendering + manual parsing."""
    start = time.time()

    # Step 1: Render JavaScript with Playwright (BS4 can't do this alone)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="networkidle")
        html = await page.content()
        await browser.close()

    # Step 2: Parse HTML manually
    soup = BeautifulSoup(html, "html.parser")

    # Step 3: Extract data with CSS selectors
    quotes = []
    for block in soup.select("div.quote"):
        text = block.select_one("span.text").get_text(strip=True)
        author = block.select_one("small.author").get_text(strip=True)
        quotes.append({"quote": text.strip("\u201c").strip("\u201d"), "author": author})

    # Step 4: Convert to markdown manually (if needed for LLMs)
    markdown = "\n".join([f"> {q['quote']}\n> -- *{q['author']}*" for q in quotes])

    return {"quotes": quotes, "markdown": markdown, "time": time.time() - start}


# Run BeautifulSoup scraper
bs_result = await scrape_with_beautifulsoup(TARGET_URL)
print(f"BeautifulSoup: {len(bs_result['quotes'])} quotes in {bs_result['time']:.2f}s")
print(f"\nSample markdown output:\n{bs_result['markdown'][:400]}...")

BeautifulSoup: 10 quotes in 1.88s

Sample markdown output:
> The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.
> -- *Albert Einstein*
> It is our choices, Harry, that show what we truly are, far more than our abilities.
> -- *J.K. Rowling*
> There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.
> -- *Albert Einstein*
...


---

## 2. Scrapy: Industrial Framework with Boilerplate

Scrapy is built for **large-scale crawling** with pipelines, middleware, and settings.
Even for a single page, you need:

- A Spider class with `parse()` method
- CrawlerProcess configuration
- Global state to capture results

**Key Limitations:**

- Scrapy's Twisted reactor conflicts with Jupyter's event loop
- Default downloader doesn't render JS (needs `scrapy-playwright`)
- More setup required for simple tasks


In [ ]:
import scrapy


# This is what a minimal Scrapy spider looks like:
class QuotesSpider(scrapy.Spider):
    """Minimal Scrapy spider - still requires class boilerplate."""

    name = "quotes"
    start_urls = [TARGET_URL]
    custom_settings = {"LOG_LEVEL": "ERROR"}

    def parse(self, response):
        for block in response.css("div.quote"):
            text = block.css("span.text::text").get("").strip("\u201c").strip("\u201d")
            author = block.css("small.author::text").get("")
            yield {"quote": text, "author": author}


# Note: Scrapy's Twisted reactor doesn't run well in Jupyter notebooks.
# In production, you would run this with: scrapy crawl quotes
# Also: This spider would find 0 quotes because Scrapy can't render JS!

print("Scrapy Spider defined (see code above)")
print("Limitations:")
print("  - Cannot run in Jupyter (Twisted reactor conflict)")
print("  - Cannot render JavaScript without scrapy-playwright")
print("  - Requires ~20 lines even for a simple page")

Scrapy Spider defined (see code above)
Limitations:
  - Cannot run in Jupyter (Twisted reactor conflict)
  - Cannot render JavaScript without scrapy-playwright
  - Requires ~20 lines even for a simple page


---

## 3. Crawl4AI: One-Line Async Extraction

Crawl4AI is designed for **AI workflows**. It handles:

- JavaScript rendering (built-in browser)
- Automatic markdown conversion
- Clean output optimized for LLMs
- Async by default

**No boilerplate, no manual parsing, no extra dependencies.**


In [ ]:
from crawl4ai import AsyncWebCrawler


async def scrape_with_crawl4ai(url: str) -> dict:
    """Crawl4AI: One async call for JS rendering + markdown output."""
    start = time.time()
    async with AsyncWebCrawler(verbose=False) as crawler:
        result = await crawler.arun(url=url)
    return {
        "markdown": result.markdown,
        "metadata": result.metadata,
        "links": result.links,
        "time": time.time() - start,
    }


# Run Crawl4AI
c4ai_result = await scrape_with_crawl4ai(TARGET_URL)
md_text = str(c4ai_result["markdown"])
print(f"Crawl4AI: {len(md_text)} chars of markdown in {c4ai_result['time']:.2f}s")
print(f"Metadata: {list(c4ai_result['metadata'].keys())}")
print(f"\nLLM-Ready Markdown Preview:\n{md_text[:600]}...")

[INIT].... → Crawl4AI 0.7.8 

[FETCH]... ↓ https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 1.07s 

[SCRAPE].. ◆ https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 0.00s 

[COMPLETE] ● https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 1.07s 

Crawl4AI: 1666 chars of markdown in 1.83s
Metadata: ['title', 'description', 'keywords', 'author']

LLM-Ready Markdown Preview:
#  [Quotes to Scrape](https://quotes.toscrape.com/)
[Login](https://quotes.toscrape.com/login)
“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”by Albert Einstein
Tags: change deep-thoughts thinking world
“It is our choices, Harry, that show what we truly are, far more than our abilities.”by J.K. Rowling
Tags: abilities choices
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”by Albert Einstein
Tags: inspirational life live miracle miracles
“The pe...
